In [14]:
import torch 
import pandas as pd
from torch import nn
import torch.nn.functional as F
from torch import optim
from protgen.models.mlp import MLP
from protgen.data import ProteinDataset
from protgen.data.make_tokens import TokenDataset
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

Load the trained MLP model

In [15]:

#Variables for the model
VOCAB_SIZE = len(special_tokens) + len(AA)
DIM_EMBEDDING = 32
NUM_HIDDEN = 128
WINDOW_SIZE = 3

#Create the MLP model 
mlp = MLP(
        vocab_size=VOCAB_SIZE,
        dim_embedding=DIM_EMBEDDING,
        num_hidden=NUM_HIDDEN,
        window_size=WINDOW_SIZE
    )

mlp.load_state_dict(torch.load("../state_dict.pt", weights_only=True))
mlp.eval()

MLP(
  (embedding): Embedding(23, 32, padding_idx=0)
  (fc1): Linear(in_features=96, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=23, bias=True)
)

Generate Protein sequence from (0,0,1)

In [16]:
#Number of parallel sequences to generate
batch_size = 50

#Maximum length of the generated sequences
max_length = 130
#Generate input windows for parallel sequences
input = torch.LongTensor([[0, 0, 1]]).repeat(batch_size, 1) # [<PAD>, <PAD>, <BOS>] starting tokens
# Track which sequences have reached EOS
finished = torch.zeros(batch_size, dtype=torch.bool)

with torch.no_grad():

    for i in range(max_length):

        # Context window size 3 (last 3tokens of every sequence)
        context = input[:, -3:]

        #size() = [batch_size, vocabulary_size]
        logits = mlp(context)

        # logits -> probabilities
        probabilities = torch.softmax(logits, dim=-1)

        # Sampling from the probabilities to get the next token for each sequence
        sampled_tokens = torch.multinomial(probabilities, num_samples=1)

        # Replace predictions with padding for already finished sequences
        next_tokens = torch.where(               # if statment 
            finished.unsqueeze(1),               # condition
            torch.full_like(sampled_tokens, 0),  # if true, tensor of zeros (padding token)
            sampled_tokens                       # if false, the sampled tokens
        )

        # Append the new tokens
        input = torch.cat((input, next_tokens), dim=1)

        # Update mask (including sequences that predicted EOS)
        finished |= next_tokens.squeeze(1).eq(2) # EOS token id is 2
        # |= Boolean or operator 

        # Stop if every sequence has finished
        if finished.all():
            break

In [17]:
print(f"Generated sequences:{input}")
print(f"Generated sequences shape: {input.shape}")

Generated sequences:tensor([[0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0],
        ...,
        [0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0]])
Generated sequences shape: torch.Size([50, 133])


Decoder 

In [18]:
def decode(tokens):
    """Convert a tensor of token ids to a string of letters

    Only the 20 residues have a character: <BOS>, <EOS>, <PAD> and the IGNORE_INDEX of a
    shifted target are dropped, so decoding either x or x_shifted gives the protein back.
    """
    total_tokens = list(special_tokens) + list(AA)
    vocab = dict(zip(total_tokens, range(len(total_tokens))))
    idx_to_vocab = {index: token for token, index in vocab.items()}
    residues = set(AA)
    return "".join(idx_to_vocab[int(token)] for token in tokens
                    if idx_to_vocab.get(int(token)) in residues)

Write sequences to results/sequences

In [19]:
sequences = [decode(seq) for seq in input]
print(f"Decoded sequences: {sequences}")
pd.DataFrame(sequences, columns=["sequence"]).to_csv("../results/sequences/generated_sequences.csv", index=False)

Decoded sequences: ['MTVTVGLSKKAKPAKITGWVKNDGSVKKMVNLPDGTSIPGFGSFKVSIMGLHARPSHALRRVEIIEQLFEQR', 'MSQDIESMVALLMLAATGNKVGLAGLSGTQENERVDGVPLDDGRVQGWAEGPRAAASVGGTSIFVKNLHELDGIQKLHARASLDRLAK', 'MTKWAPKFRQHDISVELLDVKISAVGDGTVRNLKGWVRSGRVEEVPFTTGEAVLKSFEDFRVSGWVQNLDKYGPSVDVIKRKP', 'MEYRRIRGLHARASLFEPATGASIKLDASASAKRVGRAPDVDSLPDGRVQGFDEGADVEAFVQEIREARIDGFDAQDLEEMAEFTDFGSFVKNRLSAGKTTVTVSKDGVTG', 'MNRKPAKEVEGE', 'MTANRFESRLVQLVGFRYYRVRISRSMLASGFGTFMNEQALGLTGAEPAQLANADLAGVTHVHVTGEEIEALEISGTEVEELPELILNKTQLAIQIPGFKAGKALKDSMKGK', 'MTVLNLDKLTSGTVEVIKFQEEKKGSRDGSVDIFKRDIDLWQDPHCPHARAKALNEILRSGLSRGSPSLSLGLKGKVLTSETKNISGRVQGVFFRQEGEPKEPVNK', 'MKQADHLATGETVNIVFVRASAKTVKQMLAALGLQGIGKLTEVTLADAGASVLNGFGSFNIITEISNRYELSETAGGPQRRVTTVNQKDEVPAFSAGVVQGVCGRVEWTQAVGEPIGNRVEPADPITGAE', 'MTSPGCGSKIGLHARPASKTKADLEQMSLGIIKFKPSNKEGLDALTLKLHLLELLRGFGEFIDIEGQDALAAAYVGLHARAALVSVTEGSRLTLDAKRFEVVSGKVKGQVTVGDRRFTVFIGWVENTKKD', 'MTDALDILPGFKAGLHARANASAYSLKAGSDAVFEGFGAFGAFSTNTMKIRKIRQKASPGCKIGKNMRGGGSDGFVRNEKSGDHERAARGLHARNPHLS